# 05 — Collaborative Filtering (SVD via scikit-surprise)

LightFM failed to build on Python 3.12 (Windows) — its C extension setup.py is broken on modern setuptools, and conda-forge's build caps at Python 3.11. Timeboxed, pivoted to `scikit-surprise`, which installs cleanly.

**Real trade-off, not hidden:** `scikit-surprise` doesn't support side features. Plain SVD on ratings only — no SBERT embeddings riding along to help sparse/cold-start users. Given the interaction sparsity from notebook 04, expect this to genuinely struggle for most users; that's expected, not a bug. The point of this notebook is to get an honest AUC/accuracy number so the decision to include or exclude CF in the hybrid ranker is evidence-based.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

In [ ]:
PROCESSED_DIR = Path.cwd().parent / "datasets" / "processed"
RAW_DIR = Path.cwd().parent / "datasets"

df = pd.read_parquet(PROCESSED_DIR / "feature_engineered_recipes.parquet")
interactions = pd.read_csv(RAW_DIR / "RAW_interactions.csv")

print(df.shape, interactions.shape)
print("id" in df.columns)  # must be True — re-run notebooks 01-03 if this fails

## 1. Filter interactions to recipes present in the cleaned dataset

In [ ]:
valid_recipe_ids = set(df.loc[df["source"] == "food.com", "id"].dropna())

interactions_filtered = interactions[interactions["recipe_id"].isin(valid_recipe_ids)].copy()
print(f"Interactions: {len(interactions)} -> {len(interactions_filtered)} after filtering to cleaned recipes")

# scikit-surprise needs rating scale bounds explicit
print(f"Rating range: {interactions_filtered['rating'].min()} - {interactions_filtered['rating'].max()}")

## 2. Build surprise Dataset and train/test split

In [ ]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(
    interactions_filtered["rating"].min(),
    interactions_filtered["rating"].max(),
))

surprise_data = Dataset.load_from_df(
    interactions_filtered[["user_id", "recipe_id", "rating"]], reader
)

trainset, testset = train_test_split(surprise_data, test_size=0.2, random_state=42)
print(f"Train: {trainset.n_ratings} ratings, Test: {len(testset)} ratings")
print(f"Train users: {trainset.n_users}, Train items: {trainset.n_items}")

## 3. Train SVD

In [ ]:
model = SVD(n_factors=50, n_epochs=20, random_state=42)
model.fit(trainset)

## 4. Evaluate — RMSE/MAE, then a ranking-style check

Surprise's native metrics are rating-prediction error (RMSE/MAE), not ranking metrics like AUC/precision@k. Given sparsity, low RMSE alone can be misleading (predicting the global average rating for everyone gives deceptively okay RMSE). So also compute precision@k manually — the metric that actually matters for "does this recommend the right items."

In [ ]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

In [ ]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=4.0):
    """Compute precision@k and recall@k, treating rating >= threshold as 'relevant'."""
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions, recalls = {}, {}
    for uid, ratings in user_est_true.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum(true_r >= threshold for (_, true_r) in ratings)
        n_rec_k = sum(est >= threshold for (est, _) in ratings[:k])
        n_rel_and_rec_k = sum((true_r >= threshold) for (est, true_r) in ratings[:k] if est >= threshold)

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls


precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=4.0)

avg_precision = sum(precisions.values()) / len(precisions)
avg_recall = sum(recalls.values()) / len(recalls)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")

### Reading these numbers honestly

- RMSE around 1.0-1.2 on a 1-5 scale is typical/mediocre for sparse data — not great, not catastrophic.
- **Precision@10 is the number that actually matters for your use case.** If it's low (well under content-based's ~0.75-0.85 similarity scores you saw earlier — not directly comparable metrics, but as a gut check), that supports giving CF low weight in the hybrid ranker.
- Compare this against a trivial baseline: what would precision@10 be if you just recommended each user's global most-popular items? If SVD doesn't beat that baseline, it's not adding real value over a much simpler popularity heuristic.

Report RMSE, MAE, Precision@10 back before we design the hybrid ranker's weighting.

## 5. Save model

In [ ]:
import pickle

MODELS_DIR = Path.cwd().parent / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(MODELS_DIR / "svd_model.pkl", "wb") as f:
    pickle.dump(model, f)

print(f"Saved SVD model -> {MODELS_DIR / 'svd_model.pkl'}")

## Next: `06_hybrid_ranker.ipynb`

Combine content-based (FAISS/SBERT) scores with SVD collaborative scores. Given SVD has no side-feature support and sparsity is high, expect CF's ranker weight to be low/conditional (e.g., only trusted for users above some interaction-count threshold) rather than equal to content-based — decide this from the actual precision@10 number above, not a guess.